# GCP VM MariaDB 資料庫連線語法大全

這個 Jupyter Notebook 提供使用 Python 連線到 GCP VM 上 MariaDB 資料庫的完整範例。

## 環境準備
- 需要安裝的套件：`pip install mysql-connector-python pandas sqlalchemy`
- 支援操作：連線、查詢 (SELECT)、插入 (INSERT)、更新 (UPDATE)、刪除 (DELETE)、交易處理等。
- 注意事項：
  - 確保 GCP VM 的防火牆允許 3306 連接埠。
  - 替換以下連線資訊為您自己的。
  - DATABASE 和 TABLE 可以根據需要修改。

In [ ]:
# ==================== 1. 導入必要套件 ====================
# mysql-connector-python: 用於直接連線 MariaDB
# pandas: 用於資料讀取與處理
# sqlalchemy: 用於更高階的資料庫操作（如 pandas.to_sql）
import mysql.connector
from mysql.connector import Error
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime

In [ ]:
# ==================== 2. 設定連線參數 ====================
# 請修改成您的 GCP MariaDB 資訊
# HOST: GCP VM 的外部 IP
# USER: 資料庫使用者名稱
# PASSWORD: 資料庫密碼
# DATABASE: 要操作的資料庫名稱（可根據需要修改）
# TABLE: 要操作的表格名稱（可根據需要修改）
HOST = '34.81.186.201'          # 您的 GCP 外部 IP
USER = 'datauser'
PASSWORD = '123456'
DATABASE = 'rawdata'
TABLE = '104rawdata'

# 建立 SQLAlchemy 引擎（用於 pandas 操作）
engine = create_engine(f'mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:3306/{DATABASE}')

# 測試連線是否成功
try:
    with engine.connect() as conn:
        print(f"成功連線到資料庫: {DATABASE} on {HOST}")
except Error as e:
    print(f"連線失敗: {e}")

## 範例 1: 基本連線與斷開連線

使用 `mysql.connector` 建立連線，並確保斷開連線。

In [ ]:
# ==================== 範例 1: 基本連線與斷開 ====================
try:
    # 建立連線
    conn = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        database=DATABASE
    )
    
    if conn.is_connected():
        print("連線成功！")
        cursor = conn.cursor()  # 建立游標，用於執行 SQL
        cursor.execute("SELECT VERSION()")  # 查詢 MariaDB 版本
        version = cursor.fetchone()
        print(f"MariaDB 版本: {version[0]}")
    
except Error as e:
    print(f"連線錯誤: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()  # 關閉游標
        conn.close()    # 關閉連線
        print("連線已關閉")

## 範例 2: 查詢資料 (SELECT)

- 使用直接 SQL 查詢。
- 使用 pandas 讀取查詢結果。

In [ ]:
# ==================== 範例 2.1: 使用 mysql-connector 執行 SELECT ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 查詢語法（可修改條件）
    query = f"SELECT * FROM {TABLE} LIMIT 5"  # 查詢前 5 筆資料
    cursor.execute(query)
    
    # 取得結果
    results = cursor.fetchall()
    for row in results:
        print(row)  # 印出每一行資料
    
except Error as e:
    print(f"查詢錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
# ==================== 範例 2.2: 使用 pandas 讀取 SELECT 結果 ====================
# SQL 查詢語法（可修改條件）
query = f"SELECT job_title, salary, update_date FROM {TABLE} WHERE salary LIKE '%面議%' LIMIT 10"

# 使用 pandas 直接讀取
df = pd.read_sql(query, engine)
print("查詢結果（DataFrame）：")
display(df)  # 在 Notebook 中顯示表格

## 範例 3: 插入資料 (INSERT)

- 單筆插入。
- 多筆插入。

In [ ]:
# ==================== 範例 3.1: 單筆插入 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 插入語法（假設 TABLE 有 id, job_title, salary 欄位；請根據實際 TABLE 調整）
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    data = ("測試職缺", "月薪 50,000", datetime.now().strftime('%Y-%m-%d'))  # 資料值
    
    cursor.execute(insert_query, data)
    conn.commit()  # 提交變更
    print(f"成功插入 1 筆資料，ID: {cursor.lastrowid}")
    
except Error as e:
    print(f"插入錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
# ==================== 範例 3.2: 多筆插入 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    data_list = [
        ("測試職缺1", "月薪 60,000", datetime.now().strftime('%Y-%m-%d')),
        ("測試職缺2", "月薪 70,000", datetime.now().strftime('%Y-%m-%d'))
    ]
    
    cursor.executemany(insert_query, data_list)
    conn.commit()
    print(f"成功插入 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"插入錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 4: 更新資料 (UPDATE)

更新指定條件的資料。

In [ ]:
# ==================== 範例 4: 更新資料 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 更新語法（假設有 id 欄位；請根據實際調整）
    update_query = f"UPDATE {TABLE} SET salary = %s WHERE job_title = %s"
    data = ("月薪 80,000", "測試職缺")  # 更新值與條件
    
    cursor.execute(update_query, data)
    conn.commit()
    print(f"成功更新 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"更新錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 5: 刪除資料 (DELETE)

刪除指定條件的資料。

In [ ]:
# ==================== 範例 5: 刪除資料 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 刪除語法（請小心使用，避免誤刪）
    delete_query = f"DELETE FROM {TABLE} WHERE job_title = %s"
    data = ("測試職缺",)  # 條件
    
    cursor.execute(delete_query, data)
    conn.commit()
    print(f"成功刪除 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"刪除錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 6: 交易處理 (Transaction) - Commit & Rollback

使用交易確保資料一致性。

In [ ]:
# ==================== 範例 6: 交易處理 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    conn.autocommit = False  # 關閉自動提交，啟用手動交易
    cursor = conn.cursor()
    
    # 執行多個操作
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    cursor.execute(insert_query, ("交易測試", "月薪 90,000", datetime.now().strftime('%Y-%m-%d')))
    
    # 模擬錯誤（取消註解來測試 rollback）
    # raise Error("模擬錯誤")
    
    conn.commit()  # 成功則提交
    print("交易成功提交")
    
except Error as e:
    conn.rollback()  # 錯誤則回滾
    print(f"交易失敗，回滾: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 7: 使用 pandas 寫入資料 (to_sql)

將 DataFrame 直接寫入資料庫。

In [ ]:
# ==================== 範例 7: 使用 pandas 寫入資料 ====================
# 建立一個測試 DataFrame
data = {
    'job_title': ['Pandas測試1', 'Pandas測試2'],
    'salary': ['月薪 100,000', '月薪 110,000'],
    'update_date': [datetime.now().strftime('%Y-%m-%d'), datetime.now().strftime('%Y-%m-%d')]
}
df_write = pd.DataFrame(data)

# 寫入資料庫（if_exists='append' 表示附加，不覆蓋）
df_write.to_sql(TABLE, engine, if_exists='append', index=False)
print(f"成功寫入 {len(df_write)} 筆資料到 {TABLE}")

## 範例 8: 進階查詢 - 帶參數的查詢 (防止 SQL Injection)

使用參數化查詢以提升安全性。

In [ ]:
# ==================== 範例 8: 帶參數的查詢 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # 參數化 SQL（防止 SQL Injection）
    query = f"SELECT * FROM {TABLE} WHERE job_title LIKE %s LIMIT 5"
    param = ("%測試%",)  # 查詢包含 '測試' 的職缺
    
    cursor.execute(query, param)
    results = cursor.fetchall()
    for row in results:
        print(row)
    
except Error as e:
    print(f"查詢錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 9: 建立新資料庫或表格 (CREATE)

示範建立新資料庫或表格。

In [ ]:
# ==================== 範例 9.1: 建立新資料庫 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD)
    cursor = conn.cursor()
    
    new_db = "test_database"  # 新資料庫名稱
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {new_db}")
    print(f"成功建立資料庫: {new_db}")
    
except Error as e:
    print(f"建立錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
# ==================== 範例 9.2: 建立新表格 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    new_table = "test_table"  # 新表格名稱
    create_query = f"""
        CREATE TABLE IF NOT EXISTS {new_table} (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255),
            created_at DATETIME
        )
    """
    cursor.execute(create_query)
    print(f"成功建立表格: {new_table}")
    
except Error as e:
    print(f"建立錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 注意事項

- **安全性**：避免在程式碼中硬編碼密碼，建議使用環境變數或 secrets 管理。
- **錯誤處理**：每個範例都有 try-except 來捕捉錯誤。
- **效能**：對於大量資料，使用 pandas 或 chunksize 參數來分批處理。
- **自訂**：根據您的 TABLE 結構調整 SQL 語法中的欄位名稱。